# 03. 심화 — Falcon-3 sliding regression과 scalar-carry chunk scan

이 노트북은 최근 $B$개 causal pair로 mini-batch gradient를 만드는 Falcon-3형 update와,
양의 decay를 갖는 scalar-carry affine recurrence를 chunk 단위로 결합하는 방법을
검증한다.

논문 부록의 GPU kernel이나 언어 모델 benchmark를 그대로 재현하지 않는다. 여기서는
작은 행렬에서 수식 동치·안정성·window 크기의 효과만 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(260827763)

def positive_feature(x):
    return np.where(x >= 0.0, x + 1.0, np.exp(x))

T, d, d_v = 620, 7, 2
change_point = 330
raw = rng.normal(size=(T, d))
features = positive_feature(raw)
x = np.zeros_like(features)
x[1:] = features[:-1]

A0 = rng.normal(scale=0.4, size=(d, d_v))
A1 = rng.normal(scale=0.4, size=(d, d_v))
y = np.zeros((T, d_v))
y[1:change_point] = x[1:change_point] @ A0
y[change_point:] = x[change_point:] @ A1
y[1:] += rng.normal(scale=0.12, size=(T - 1, d_v))

## 1. 최근 $B$개 pair를 쓰는 Falcon-3형 update

window $\mathcal{I}_t$의 feature matrix를 $X_t$, target matrix를 $Y_t$라 하고
$\mu_t=\lambda_{\max}(X_t^\top X_t/B)$로 둔다.

$$\eta_t=\frac{\beta}{\mu_t+\lambda+\epsilon},\qquad
S_t=(1-\eta_t\lambda)S_{t-1}+\frac{\eta_t}{B}
X_t^\top(Y_t-X_tS_{t-1}).$$

논문의 정확한 kernel 최적화 대신 정의를 그대로 읽기 쉬운 NumPy로 구현한다.

In [ ]:
def falcon3_sliding(x, y, window, beta=0.8, ridge=0.02, eps=1e-8):
    state = np.zeros((x.shape[1], y.shape[1]))
    errors = np.zeros(len(x))
    state_norms = np.zeros(len(x))
    steps = np.zeros(len(x))

    for t in range(len(x)):
        prediction = state.T @ x[t]
        errors[t] = np.mean((y[t] - prediction) ** 2)

        start = max(1, t - window + 1)
        X = x[start : t + 1]
        Y = y[start : t + 1]
        if len(X) == 0:
            state_norms[t] = np.linalg.norm(state)
            continue

        covariance = (X.T @ X) / len(X)
        mu = max(float(np.linalg.eigvalsh(covariance)[-1]), 0.0)
        eta = beta / (mu + ridge + eps)
        residuals = Y - X @ state
        state = (1.0 - eta * ridge) * state + (eta / len(X)) * X.T @ residuals

        steps[t] = eta
        state_norms[t] = np.linalg.norm(state)
    return errors, state_norms, steps

windows = [1, 4, 8]
runs = {B: falcon3_sliding(x, y, B) for B in windows}
for B in windows:
    errors, norms, steps = runs[B]
    print(
        f"B={B}: pre-shift MSE={errors[60:change_point].mean():.5f}, "
        f"post-shift MSE={errors[change_point + 60:].mean():.5f}, "
        f"max ||S||={norms.max():.3f}"
    )
    assert np.all(np.isfinite(errors))
    assert np.all(np.isfinite(norms))
    assert np.all(steps >= 0.0)

In [ ]:
def moving_average(values, window=25):
    return np.convolve(values, np.ones(window) / window, mode="same")

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for B, (errors, norms, _) in runs.items():
    axes[0].semilogy(moving_average(errors), label=f"window B={B}")
    axes[1].plot(norms, label=f"window B={B}")
for ax in axes:
    ax.axvline(change_point, color="black", linestyle="--")
    ax.legend()
axes[0].set(title="sliding regression error", ylabel="moving MSE")
axes[1].set(title="fast-weight state norm", xlabel="time", ylabel="Frobenius norm")
fig.tight_layout()
plt.show()

## 2. 양의 decay와 affine recurrence

$S_t=\gamma_tS_{t-1}+U_t$에서 $\gamma_t<0$이어도 일반 affine scan 자체는
정의되지만 부호 반전이 생기고, 실수 $\log\gamma_t$를 쓰는 논문의 log-space
unrolling/renormalization은 적용할 수 없다. positive-decay renormalization의 핵심은
$\alpha_t=\min(\eta_t\lambda_t,1-\epsilon_\gamma)$,
$\gamma_t=1-\alpha_t$로 두어 $\gamma_t\ge\epsilon_\gamma$를 보장하는 것이다.

In [ ]:
eta_demo = np.array([0.2, 0.8, 1.4, 3.0])
lambda_demo = np.array([0.1, 0.7, 0.9, 0.8])
gamma_naive = 1.0 - eta_demo * lambda_demo
eps_gamma = 1e-4
alpha_safe = np.minimum(eta_demo * lambda_demo, 1.0 - eps_gamma)
gamma_safe = 1.0 - alpha_safe

print("naive gamma:", gamma_naive)
print("safe gamma :", gamma_safe)
assert np.any(gamma_naive < 0.0)
assert np.all(gamma_safe >= eps_gamma - 1e-12)

## 3. 순차 recurrence와 scalar-carry chunk affine scan의 동치

여기서는 full Falcon-3 rank-$B$ transition보다 단순한 scalar-carry 부분 문제를
분리한다. 한 chunk의 여러 update는 다시 하나의 affine map
$S_{out}=A_{chunk}S_{in}+B_{chunk}$로 합칠 수 있다. 아래 코드는 각 chunk 내부의
`A`, `B`를 만든 뒤 경계 상태에 적용한다. 이 작은 검사는 parallel kernel을 구현하기 전
가장 먼저 통과해야 할 algebra test다. Falcon-3의 `tensorInv`나 rank-$B$
ParallelFlow 전체를 검증하는 코드는 아니다.

In [ ]:
def sequential_affine(gamma, updates, initial):
    state = initial.copy()
    states = []
    for g_t, u_t in zip(gamma, updates):
        state = g_t * state + u_t
        states.append(state.copy())
    return np.stack(states)

def chunk_affine(gamma, updates, initial, chunk_size):
    state = initial.copy()
    states = []
    for start in range(0, len(gamma), chunk_size):
        stop = min(start + chunk_size, len(gamma))
        # prefix affine maps inside this chunk: local_state = A * state + B
        A = 1.0
        B = np.zeros_like(state)
        local_maps = []
        for g_t, u_t in zip(gamma[start:stop], updates[start:stop]):
            A = g_t * A
            B = g_t * B + u_t
            local_maps.append((A, B.copy()))
        incoming = state.copy()
        states.extend(A_i * incoming + B_i for A_i, B_i in local_maps)
        state = A * incoming + B
    return np.stack(states)

length, matrix_dim = 37, 4
raw_eta = rng.uniform(0.0, 2.0, size=length)
raw_lambda = rng.uniform(0.0, 1.2, size=length)
gamma = 1.0 - np.minimum(raw_eta * raw_lambda, 1.0 - eps_gamma)
updates = rng.normal(scale=0.1, size=(length, matrix_dim, matrix_dim))
initial = rng.normal(scale=0.1, size=(matrix_dim, matrix_dim))

states_seq = sequential_affine(gamma, updates, initial)
states_chunk = chunk_affine(gamma, updates, initial, chunk_size=8)
max_scan_error = np.max(np.abs(states_seq - states_chunk))
print(f"sequential vs chunk maximum error: {max_scan_error:.3e}")

np.testing.assert_allclose(states_seq, states_chunk, rtol=1e-12, atol=1e-12)
assert np.all(gamma > 0.0)

## 4. 결과를 논문 주장과 혼동하지 않기

- `B` sweep은 작은 synthetic online regression의 분산·적응성 절충만 보여준다.
- 논문의 FineWeb-Edu 언어 모델 및 33–48자리 덧셈 결과를 재현하려면 공개 코드의
  tokenizer, model configuration, 약 49.2B-token 학습 조건, GPU kernel과 평가 harness가 필요하다.
- 실제 kernel 검증에는 recurrent/parallel forward 동치뿐 아니라 gradient check,
  mixed-precision stress test, chunk 경계와 `x[0]=0` boundary test가 필요하다.
- 논문 표와 이 노트북의 수치를 직접 비교하지 않는다.

다음 확장 과제: `beta`, `ridge`, `B`, 변화 속도를 grid search하고 평균뿐 아니라
change point 직후 회복 시간과 최대 state norm을 함께 보고하라.